Question 1

Using homework_8.1.csv, find the Average treatment effect with inverse probability weighting. Then, include your code and a written explanation of your work (mentioning any choices or strategies you made in writing the code) in your homework reflection.  



Here are some steps to follow: 



* Estimate the propensity scores using logistic regression. Fit the model so that the Z values predict ﻿X﻿. 

* Use the model to predict the propensity scores (e.g., using predict_proba if you are using sklearn). 

* Calculate inverse probability weights (﻿1 over P﻿ for ﻿X equals 1﻿ and ﻿fraction numerator 1 over denominator 1 minus P end fraction﻿ for ﻿X equals 0﻿). 

* Estimate the average treatment effect (the Y difference between ﻿X equals 1﻿ and ﻿X equals 0﻿, using the appropriate weights for each). 



In [24]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt



In [25]:
df = pd.read_csv('/Users/gkumargaur/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_8.1.csv')

print('Total number of rows in the dataset:', len(df))
df.head()

Total number of rows in the dataset: 1000


,X,Y,Z
0,1,4.109218,1.764052
1,0,2.259504,0.400157
2,0,-0.647584,0.978738
3,0,2.106071,2.240893
4,1,3.583464,1.867558


In [26]:
from sklearn.linear_model import LogisticRegression

# Step 1: propensity model -- Z predicts treatment X
# C=np.inf -> plain (unpenalized) logistic regression, so sklearn's default L2 shrinkage doesn't bias p
ps_model = LogisticRegression(C=np.inf)
ps_model.fit(df[['Z']], df['X'])

# Comparison model: sklearn's DEFAULT (L2 penalty, C=1.0) -- used later to check the choice doesn't matter
ps_model_default = LogisticRegression().fit(df[['Z']], df['X'])
print(f"Z coefficient, unpenalized: {ps_model.coef_[0][0]:.4f}")
print(f"Z coefficient, default C=1: {ps_model_default.coef_[0][0]:.4f}")



Z coefficient, unpenalized: 0.9724
Z coefficient, default C=1: 0.9658


In [27]:
# Step 2: propensity scores gives each person's probability of being treated p = P(X=1 | Z)
df['p'] = ps_model.predict_proba(df[['Z']])[:, 1]
p_default = ps_model_default.predict_proba(df[['Z']])[:, 1]

print(f"Propensity score range: {df['p'].min():.3f} - {df['p'].max():.3f}")



Propensity score range: 0.047 - 0.933


In [28]:
# Step 3: inverse probability weights: 1/p for treated people(X=1), 1/(1-p) for untreated people(X=0)
df['ipw'] = np.where(df['X'] == 1, 1 / df['p'], 1 / (1 - df['p']))
w_default = np.where(df['X'] == 1, 1 / p_default, 1 / (1 - p_default))

print(f"Max weight: {df['ipw'].max():.1f}")



Max weight: 12.9


In [29]:
# Step 4: ATE = weighted mean Y (treated) - weighted mean Y (untreated)
treated, control = df[df['X'] == 1], df[df['X'] == 0]
y1 = np.average(treated['Y'], weights=treated['ipw'])
y0 = np.average(control['Y'], weights=control['ipw'])
ate_ipw = y1 - y0

naive = treated['Y'].mean() - control['Y'].mean()

# Alternative 1: unnormalized (Horvitz-Thompson) IPW: mean(X*Y/p) - mean((1-X)*Y/(1-p))
X, Y, p = df['X'], df['Y'], df['p']
ate_ht = np.mean(X * Y / p) - np.mean((1 - X) * Y / (1 - p))

# Alternative 2: same normalized formula, but weights from the default C=1.0 model
t, c = (df['X'] == 1).values, (df['X'] == 0).values
ate_default = (np.average(df['Y'][t], weights=w_default[t])
               - np.average(df['Y'][c], weights=w_default[c]))

print(f"Naive difference in means:                {naive:.3f}")
print(f"IPW ATE estimate (main):                  {ate_ipw:.3f}")
print(f"IPW ATE, unnormalized (Horvitz-Thompson): {ate_ht:.3f}")
print(f"IPW ATE, sklearn default C=1.0:           {ate_default:.3f}")

Naive difference in means:                3.037
IPW ATE estimate (main):                  2.268
IPW ATE, unnormalized (Horvitz-Thompson): 2.236
IPW ATE, sklearn default C=1.0:           2.274


## What we evaluated
1. **Propensity model:** a logistic regression where Z predicts X (being treated).
2. **Propensity scores:** `predict_proba(...)[:, 1]` gives each person's p = P(X=1 | Z).
3. **Weights:** 1/p for treated people (X=1) and 1/(1−p) for untreated people (X=0).
4. **ATE:** the weighted average of Y for the treated minus the weighted average of Y for the untreated.

## What assumtions has been made
- **I turned off scikit-learn's default regularization (`C=np.inf`).** By default, `LogisticRegression` slightly shrinks the coefficients, which pulls the propensity scores toward 0.5. We want the plain probability of treatment here. In this dataset it hardly matters: the default setting gives 2.274.
- **I used weighted means divided by the sum of the weights.** `np.average` does this, and it is called the normalized (Hájek) estimator. The other version, `mean(X·Y/p) − mean((1−X)·Y/(1−p))`, gives 2.236. The normalized version is more stable when a few weights are large.
- **I checked overlap.** All scores fall between 0.05 and 0.93, and the largest weight is about 13. So no single person dominates the estimate, and I didn't need to trim extreme scores.
- **Interpretation:** the simple difference of 3.04 overstates the effect. Z raises both the chance of treatment (logit coefficient about 0.97) and Y, so treated people would have had higher Y anyway. Reweighting makes the treated and untreated groups look alike on Z, which removes that bias and gives an ATE of about 2.27. This rests on one assumption: Z is the only confounder.

---
Question 2

The propensity scores of the first three items in the dataframe are closest to: 

In [30]:
print(df['p'].head(3))

0    0.841703
1    0.585329
2    0.712449
Name: p, dtype: float64


---
Question 3
Using homework_8.2.csv, match all treated items to the single nearest untreated item using the Mahalanobis distance. (Do this with replacement — the same untreated item can be used again.) 



* Use the Mahalanobis function from scipy.spatial.distance 

* For the inverse covariance matrix, use all ﻿Z 1﻿ values and all ﻿Z 2﻿ values, make them into a ﻿2 x N﻿ matrix, find its ﻿2 x 2﻿covariance, and invert. 



In [31]:
df2 = pd.read_csv('/Users/gkumargaur/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_8.2.csv')

print('Total number of rows in the dataset:', len(df2))
df2.head()

Total number of rows in the dataset: 1000


,X,Y,Z1,Z2
0,1,4.652085,1.764052,2.320015
1,1,2.590221,0.400157,1.292631
2,1,3.898981,0.978738,0.556423
3,1,5.857179,2.240893,2.345607
4,1,3.647489,1.867558,2.095611


In [32]:
# Step1 : Inverse covariance matrix
from scipy.spatial.distance import mahalanobis

Z = np.vstack((df2['Z1'], df2['Z2'])) #shape (2, 1000)
# np.cov treats each row as a variable. That's why the matrix must be 2 × N; 
# if you pass N × 2, you get a 1000 × 1000 matrix.
# Use all 1,000 rows, treated and untreated together, as the question says.
cov = np.cov(Z) #shape (2, 2)

inv_cov = np.linalg.inv(cov) #inverse covariance

print('Z.shape, cov.shape', Z.shape, cov.shape)
print('inv_cov', inv_cov)


Z.shape, cov.shape (2, 1000) (2, 2)
inv_cov [[ 2.02734407 -1.03387633]
 [-1.03387633  1.06684813]]


In [33]:
# Step2 : Split into treated and untreated
treated = df2[df2['X'] == 1]
untreated = df2[df2['X'] == 0]

T = treated[['Z1', 'Z2']].values   # NumPy arrays make the loop fast
C = untreated[['Z1', 'Z2']].values


In [34]:

# Step3 : For each treated row, find the nearest untreated row

match_idx = []
for t in T:
    dists = [mahalanobis(t, c, inv_cov) for c in C]   # distance to every control
    match_idx.append(np.argmin(dists))           # position of the closest one

# With replacement comes for by default. You search all controls every time and never remove one, 
# so the same control can be picked again.

In [36]:
# Step 4: Estimate the effect (the ATT)

matched_Y = untreated['Y'].values[match_idx]
att = np.mean(treated['Y'].values - matched_Y)
print('Simple difference in means : ',treated['Y'].mean() - untreated['Y'].mean())
print(f"Matched ATT: {att:.3f}")

Simple difference in means :  4.047942450854534
Matched ATT: 3.438


**Important observations:**
- **Why Mahalanobis distance:** it adjusts for the scale of Z1 and Z2 and for the correlation between them, so neither variable dominates the distance.
- **Why with replacement:** it gives closer matches (less bias) but higher variance, since a few controls are reused many times.
- **Why the effect drops from 4.05 to 3.44:** the treated rows have higher Z values, and matching removes that difference.

---
Question 4

Find the nearest Z1 and Z2 values of the treated item with the least common support (the farthest Mahalanobis distance from the untreated). 

In [38]:
# Common support asks whether a treated row has any similar untreated rows. The treated row with 
# the least common support is the one whose closest untreated match is still the farthest away.

match_idx = []
match_dist = []                                  # new: store the nearest distance too
for t in T:
    dists = [mahalanobis(t, c, inv_cov) for c in C]
    match_idx.append(np.argmin(dists))
    match_dist.append(np.min(dists))             # new: store the nearest distance too

# Find the treated row with the least common support
match_dist = np.array(match_dist)
worst = np.argmax(match_dist)                    # position within the treated rows

print("Farthest nearest-match distance:", match_dist[worst])
print(treated.iloc[worst])                       # this row's Z1 and Z2

nearest = untreated.iloc[match_idx[worst]]
print("Treated item:      ", treated.iloc[worst][['Z1', 'Z2']].values)
print("Nearest untreated: ", nearest[['Z1', 'Z2']].values)   # -> [1.52, -1.28]


Farthest nearest-match distance: 1.3830045330353775
X     1.000000
Y     6.540236
Z1    2.696224
Z2    0.538155
Name: 494, dtype: float64
Treated item:       [2.69622405 0.53815549]
Nearest untreated:  [ 1.51999486 -1.28220794]
